In [1]:
import pandas as pd

orders = pd.read_csv("../data/raw/orders.csv")
products = pd.read_csv("../data/raw/products.csv")
customers = pd.read_csv("../data/raw/customers.csv")
returns = pd.read_csv("../data/raw/returns.csv")
# Display the first few rows of each DataFrame to verify successful loading
print("Orders DataFrame:")
print(orders.head())
print("\nProducts DataFrame:")
print(products.head())
print("\nCustomers DataFrame:")
print(customers.head())
print("\nReturns DataFrame:")
print(returns.head())

Orders DataFrame:
  order_id customer_id product_id  order_date  quantity  price  revenue  \
0   O00001       C0349        P34  2024-03-15         1  973.0    973.0   
1   O00002       C0076        P33  2024-12-08         2  891.0   1782.0   
2   O00003       C0079        P53  2024-03-22         1  212.0    212.0   
3   O00004       C0413        P28  2024-07-16         2  471.0    942.0   
4   O00005       C0069        P24  2024-03-04         1  755.0    755.0   

        city  
0     Mumbai  
1  Ahmedabad  
2  Hyderabad  
3       Pune  
4  Ahmedabad  

Products DataFrame:
  product_id                                       product_name  \
0         P1                     Austin Hot & Cold Water Bottle   
1         P2                 Crypto Art Hot & Cold Water Bottle   
2         P3  Pexpo Amaze Thermo Steel Bottle | Tri-Ply Vacu...   
3         P4              Craft SS Water Bottle with Sipper Cap   
4         P5                  Piano Art Hot & Cold Water Bottle   

         raw_pric

In [2]:
products.info()
products.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_id    62 non-null     object
 1   product_name  62 non-null     object
 2   raw_price     62 non-null     object
 3   collection    62 non-null     object
 4   product_url   62 non-null     object
dtypes: object(5)
memory usage: 2.6+ KB


product_id      0
product_name    0
raw_price       0
collection      0
product_url     0
dtype: int64

In [3]:
products.describe()

,product_id,product_name,raw_price,collection,product_url
count,62,62,62,62,62
unique,62,46,32,4,47
top,P1,Piano Art Hot & Cold Water Bottle,₹ 891.00,Best Sellers,https://pexpo.in/products/piano-artful-vacuum-...
freq,1,3,6,16,3


In [4]:
import re

def extract_min_price(price_text):
    if pd.isna(price_text):
        return None
    price_text = price_text.replace(",", "")
    nums = re.findall(r"\d+\.?\d*", price_text)
    return float(nums[0]) if nums else None

products["price"] = products["raw_price"].apply(extract_min_price)


In [5]:
def infer_product_type(name):
    name = name.lower()
    if "bottle" in name or "flask" in name:
        return "Bottle"
    elif "lunch" in name or "tiffin" in name:
        return "Lunch Box"
    elif "kids" in name:
        return "Kids Product"
    else:
        return "Accessories"

products["product_type"] = products["product_name"].apply(infer_product_type)


In [6]:
products["product_line"] = products["product_name"].apply(lambda x: x.split(" ")[0])


In [7]:
products_cleaned = products[[
    "product_id",
    "product_name",
    "product_type",
    "product_line",
    "price",
    "collection",
    "product_url"
]]
products_cleaned.head()

,product_id,product_name,product_type,product_line,price,collection,product_url
0,P1,Austin Hot & Cold Water Bottle,Bottle,Austin,905.0,Best Sellers,https://pexpo.in/products/austin
1,P2,Crypto Art Hot & Cold Water Bottle,Bottle,Crypto,806.0,Best Sellers,https://pexpo.in/products/crypto-art-vacuum-flask
2,P3,Pexpo Amaze Thermo Steel Bottle | Tri-Ply Vacu...,Bottle,Pexpo,1019.0,Best Sellers,https://pexpo.in/products/amaze-vacuum-bottle
3,P4,Craft SS Water Bottle with Sipper Cap,Bottle,Craft,403.0,Best Sellers,https://pexpo.in/products/craft-ss-water-bottl...
4,P5,Piano Art Hot & Cold Water Bottle,Bottle,Piano,755.0,Best Sellers,https://pexpo.in/products/piano-artful-vacuum-...


In [8]:
customers.info()
customers.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  500 non-null    object
 1   city         500 non-null    object
 2   gender       500 non-null    object
 3   age_group    500 non-null    object
dtypes: object(4)
memory usage: 15.8+ KB


customer_id    0
city           0
gender         0
age_group      0
dtype: int64

In [9]:
customers = customers.dropna(subset=["city"])

customers["gender"] = customers["gender"].fillna("Unknown")
customers["age_group"] = customers["age_group"].fillna("Unknown")


In [10]:
for col in ["city", "gender", "age_group"]:
    customers[col] = customers[col].astype("category")


In [11]:
orders.info()
orders.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     3000 non-null   object 
 1   customer_id  3000 non-null   object 
 2   product_id   3000 non-null   object 
 3   order_date   3000 non-null   object 
 4   quantity     3000 non-null   int64  
 5   price        3000 non-null   float64
 6   revenue      3000 non-null   float64
 7   city         3000 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 187.6+ KB


order_id       0
customer_id    0
product_id     0
order_date     0
quantity       0
price          0
revenue        0
city           0
dtype: int64

In [12]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["quantity"] = orders["quantity"].astype(int)
orders["price"] = orders["price"].astype(float)


In [13]:
orders = orders.dropna(subset=["city"])
orders["revenue"] = orders["quantity"] * orders["price"]


In [14]:
orders = orders[orders["quantity"] > 0]
orders = orders[orders["price"] > 0]


In [15]:
orders["year"] = orders["order_date"].dt.year
orders["month"] = orders["order_date"].dt.month
orders["day"] = orders["order_date"].dt.day


In [16]:
returns.info()
returns.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   order_id       300 non-null    object
 1   return_reason  300 non-null    object
dtypes: object(2)
memory usage: 4.8+ KB


order_id         0
return_reason    0
dtype: int64

In [17]:
returns["return_reason"] = returns["return_reason"].fillna("Unknown")


In [18]:
products.to_csv("../data/cleaned/products_cleaned.csv", index=False)
orders.to_csv("../data/cleaned/orders_cleaned.csv", index=False)
customers.to_csv("../data/cleaned/customers_cleaned.csv", index=False)
returns.to_csv("../data/cleaned/returns_cleaned.csv", index=False)
print("Cleaned data saved successfully.")


Cleaned data saved successfully.
